In [ ]:
# Install dependencies once in the selected kernel, then restart it:
# %pip install -r requirements.txt
# For Parakeet: %pip install -r requirements-nemo.txt
import os
import shutil
import sys
from pathlib import Path
from dotenv import load_dotenv

# Match pythonChatAgent.py before importing torch. Restart the kernel after changes.
CPU_THREADS = 8
os.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(CPU_THREADS))

load_dotenv(dotenv_path=Path.cwd() / ".env", override=True)

# Keep caches portable even if VS Code supplied an old machine's notebook.env.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
os.environ["XDG_CACHE_HOME"] = str(Path.cwd() / ".cache")
os.environ["HF_HOME"] = str(Path.cwd() / ".cache" / "huggingface")
os.environ["NEMO_CACHE_DIR"] = str(Path.cwd() / ".cache" / "nemo")
if not shutil.which("ffmpeg"):
    # Whisper invokes ffmpeg by name; provision the bundled binary in this environment.
    import imageio_ffmpeg
    ffmpeg_path = Path(sys.executable).parent / ("ffmpeg.exe" if os.name == "nt" else "ffmpeg")
    shutil.copy2(imageio_ffmpeg.get_ffmpeg_exe(), ffmpeg_path)
print(f"Python: {sys.executable}")
print("Dependencies and FFmpeg ready. Use a fresh kernel after installing packages.")


In [ ]:
import cProfile
import contextlib
import gc
import importlib.metadata
import io
import json
import math
import platform
import pstats
import queue
import re
import tempfile
import threading
import time
import wave

import numpy as np
import psutil
import soundfile as sf
import torch
import whisper
import pandas as pd
from IPython.display import display
from words import get_keywords_for_audio_type

AUDIO_DIR = "./selected_audio"
OUTPUT_CSV = "./model_comparison.csv"
ATTEMPTS_PER_AUDIO = 5
LANGUAGE = "en"
BATCH_STATE = "decision"  # Match the script's default; also supports reason/default.
MIN_AUDIO_SEC = 0.15
MAX_AUDIO_SECONDS = 30
REQUEST_TIMEOUT_SECONDS = 10.0  # Actual script timeout, despite its "OVER 5 SECONDS" text.
PROFILE_REQUESTS = True  # The script includes cProfile/report printing in request latency.
RAISE_WINDOWS_PRIORITY = True

# The script uses base for stock Whisper and this exact repo for fine-tuned Whisper.
# Keep small as an additional experiment; it is a different model from base.
MODELS = [
    {"name": "base", "backend": "whisper", "model": "base"},
    {"name": "small", "backend": "whisper", "model": "small"},
    {"name": "whisper_sm_finetuned", "backend": "huggingface", "model": "norillalearning/whisper_small_lora_merged"},
    {"name": "whisper_sm_finetuned_museum", "backend": "huggingface", "model": "Norillalearning15/whisper-small-museum"},
    {"name": "parakeet_v3", "backend": "nemo", "model": "nvidia/parakeet-tdt-0.6b-v3", "pretrained": True},
    {"name": "parakeet_finetuned_museum", "backend": "nemo", "model": "~/Downloads/parakeet_adapter_museum_best.nemo"},
    {"name": "parakeet_finetuned", "backend": "nemo", "model": "norillalearning/parakeet_finetuned", "pretrained": True, "filename": "parakeet_fullft_finetuned.nemo"},
]
MODEL_CACHE = Path(".cache").resolve()
ALLOWED_CHARS = re.compile(r"[^a-zA-Z0-9\s.,!?'\-]")


def clean_transcript(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", ALLOWED_CHARS.sub("", text)).strip()


def get_audio_duration_seconds(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except Exception:
        return None


def wav_duration_seconds(path):
    try:
        with contextlib.closing(wave.open(str(path), "rb")) as wav:
            rate = wav.getframerate()
            return wav.getnframes() / float(rate) if rate else 0.0
    except Exception:
        return None


def configure_runtime():
    torch.set_num_threads(CPU_THREADS)
    priority = "unchanged"
    if RAISE_WINDOWS_PRIORITY and os.name == "nt":
        try:
            psutil.Process(os.getpid()).nice(psutil.HIGH_PRIORITY_CLASS)
            priority = "HIGH_PRIORITY_CLASS"
        except Exception as exc:
            priority = f"Could not raise process priority: {exc}"
    versions = {}
    for package in ("torch", "openai-whisper", "transformers", "huggingface-hub",
                    "nemo-toolkit", "librosa", "numpy", "soundfile", "psutil"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return {
        "reference": "pythonChatAgent.py",
        "platform": platform.platform(), "python": platform.python_version(),
        "python executable": sys.executable, "packages": versions,
        "torch threads": torch.get_num_threads(),
        "OMP_NUM_THREADS": os.environ.get("OMP_NUM_THREADS"),
        "MKL_NUM_THREADS": os.environ.get("MKL_NUM_THREADS"),
        "process priority": priority, "cuda available": torch.cuda.is_available(),
        "audio type": BATCH_STATE, "request profiling": PROFILE_REQUESTS,
        "request timeout [s]": REQUEST_TIMEOUT_SECONDS,
        "timing scope": "request metadata checks, worker/profiler, audio loading, features, inference, decoding, cleanup and confidence; excludes model loading, warm-up and intent classification",
        "models": {},
    }


def load_whisper_component(component, model_id, **options):
    """Handle older list-valued tokenizer exports without mutating the checkpoint."""
    try:
        return component.from_pretrained(model_id, **options)
    except AttributeError as exc:
        if "'list' object has no attribute 'keys'" not in str(exc):
            raise
        config_path = Path(model_id) / "tokenizer_config.json"
        if not config_path.is_file():
            from huggingface_hub import hf_hub_download
            config_path = Path(hf_hub_download(model_id, "tokenizer_config.json", **options))
        special_tokens = json.loads(config_path.read_text()).get("extra_special_tokens")
        if not isinstance(special_tokens, list):
            raise
        return component.from_pretrained(
            model_id, **options, extra_special_tokens={}, additional_special_tokens=special_tokens,
        )


def get_model(spec):
    print(f"Loading {spec['name']}: {spec['backend']} / {spec['model']}...")
    if spec["backend"] == "whisper":
        # Like load_base_whisper(): Whisper chooses CUDA if available, otherwise CPU.
        return whisper.load_model(spec["model"], download_root=str(MODEL_CACHE / "whisper"))
    if spec["backend"] == "huggingface":
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        from transformers import WhisperFeatureExtractor, WhisperTokenizer

        token = os.environ.get("HF_TOKEN") or None
        cache_dir = str(MODEL_CACHE / "huggingface")
        processor_id = spec.get("processor", spec["model"])
        options = {"cache_dir": cache_dir, "token": token}
        try:
            processor = load_whisper_component(WhisperProcessor, processor_id, **options)
        except OSError:
            # Same fallback as the script for merged exports without preprocessing files.
            extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small", **options)
            tokenizer = load_whisper_component(WhisperTokenizer, processor_id, **options)
            processor = WhisperProcessor(feature_extractor=extractor, tokenizer=tokenizer)
        model = WhisperForConditionalGeneration.from_pretrained(spec["model"], **options)
        model.eval()
        return {"model": model, "processor": processor}
    if spec["backend"] == "nemo":
        if not spec.get("pretrained", False):
            checkpoint = Path(spec["model"]).expanduser().resolve()
            if checkpoint.suffix.lower() != ".nemo" or not checkpoint.is_file():
                raise FileNotFoundError(f"NeMo checkpoint not found: {checkpoint}")
        try:
            from nemo.collections.asr.models import ASRModel
        except ImportError as exc:
            raise ImportError(f"NeMo ASR import failed: {exc}. Check dependency versions in this kernel.") from exc
        if spec.get("pretrained", False):
            if spec.get("filename"):
                from huggingface_hub import hf_hub_download
                checkpoint = Path(hf_hub_download(
                    repo_id=spec["model"], filename=spec["filename"],
                    token=os.environ.get("HF_TOKEN") or None,
                ))
            else:
                checkpoint = Path(ASRModel.from_pretrained(model_name=spec["model"], return_model_file=True))
        # Preserve support for adapter exports without modifying their checkpoint files.
        config = ASRModel.restore_from(restore_path=str(checkpoint), return_config=True)
        if config.get("adapters") and config.get("encoder", {}).get("_target_"):
            from nemo.core.classes.mixins import adapter_mixins
            from omegaconf import open_dict
            adapter_info = adapter_mixins.get_registered_adapter(config.encoder._target_)
            if adapter_info is not None and config.encoder._target_ != adapter_info.adapter_class_path:
                with open_dict(config):
                    config.encoder._target_ = adapter_info.adapter_class_path
        model = ASRModel.restore_from(restore_path=str(checkpoint), override_config_path=config, strict=True)
        model.eval()
        if model.cfg.get("preprocessor", {}).get("sample_rate", 16000) != 16000:
            raise ValueError("The reference expects a 16 kHz NeMo checkpoint.")
        return model
    raise ValueError(f"Unsupported backend: {spec['backend']}")


def transcribe_audio(model, spec, audio_file, audio_type):
    """Match pythonChatAgent.run_transcribe, including file loading and confidence."""
    if spec["backend"] == "nemo":
        with torch.no_grad():
            hypotheses = model.transcribe([str(audio_file)], return_hypotheses=True)
        hyp = hypotheses[0]
        if isinstance(hyp, (list, tuple)):
            hyp = hyp[0]
        text = getattr(hyp, "text", None)
        if not isinstance(text, str):
            text = "NULL result data not of type String"
        score = getattr(hyp, "score", None)
        return clean_transcript(text), round(float(score), 4) if score is not None else 0.0
    if spec["backend"] == "huggingface":
        import librosa
        hf_model, processor = model["model"], model["processor"]
        audio_arr, _sr = librosa.load(str(audio_file), sr=16000)
        inputs = processor(audio_arr, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            generated = hf_model.generate(
                inputs["input_features"], language=LANGUAGE, task="transcribe",
                return_dict_in_generate=True, output_scores=True,
            )
        text = processor.batch_decode(generated.sequences, skip_special_tokens=True)[0]
        text = clean_transcript(text if isinstance(text, str) else "")
        try:
            scores = torch.stack(generated.scores, dim=1)
            log_probs = torch.log_softmax(scores, dim=-1)
            tokens = generated.sequences[:, 1:1 + log_probs.shape[1]]
            token_logprobs = log_probs.gather(-1, tokens.unsqueeze(-1)).squeeze(-1)
            confidence = round(float(np.exp(token_logprobs.mean().item())), 4)
        except Exception:
            confidence = 0.0
        return text, confidence
    keywords = get_keywords_for_audio_type(audio_type)
    prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(keywords)
    print(f"INITIAL PROMPT [{audio_type}]: {prompt}", flush=True)
    result = model.transcribe(
        str(audio_file), initial_prompt=prompt, language=LANGUAGE,
        fp16=False, condition_on_previous_text=False,
    )
    text = result.get("text") if isinstance(result, dict) else None
    if not isinstance(text, str):
        text = "NULL result data not have any text string"
    segments = result.get("segments", []) if isinstance(result, dict) else []
    confidence = round(math.exp(sum(s["avg_logprob"] for s in segments) / len(segments)), 4) if segments else 0.0
    return clean_transcript(text), confidence


def warmup_model(model, spec):
    """The reference warms HF Whisper/Parakeet with 4 seconds of silence, not stock Whisper."""
    if spec["backend"] == "whisper":
        return 0.0
    started = time.perf_counter()
    silence = np.zeros(16000 * 4, dtype=np.float32)
    if spec["backend"] == "huggingface":
        inputs = model["processor"](silence, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            model["model"].generate(inputs["input_features"], language=LANGUAGE, task="transcribe")
    else:
        with tempfile.TemporaryDirectory(prefix="whisper-warmup-") as directory:
            warmup_path = Path(directory) / "_warmup.wav"
            sf.write(warmup_path, silence, 16000)
            with torch.no_grad():
                model.transcribe([str(warmup_path)], return_hypotheses=True)
    return (time.perf_counter() - started) * 1000


def run_request(model, spec, audio_file, audio_type):
    """Measure the reference request path. Drain a timeout's worker before the next attempt."""
    started = time.perf_counter()
    duration = get_audio_duration_seconds(audio_file) or wav_duration_seconds(audio_file)
    if not Path(audio_file).is_file() or (duration is not None and duration < MIN_AUDIO_SEC):
        return "", 0.0, (time.perf_counter() - started) * 1000, "skipped", "Missing or too-short audio"
    keywords = get_keywords_for_audio_type(audio_type)
    print(f"KEYWORDS [{audio_type}]: {', '.join(keywords)} (informational unless stock Whisper uses them)", flush=True)
    results, completed = queue.Queue(), threading.Event()

    def worker():
        try:
            duration = get_audio_duration_seconds(audio_file)
            if duration is not None and duration > MAX_AUDIO_SECONDS:
                raise ValueError(f"Audio duration {duration:.1f}s exceeds {MAX_AUDIO_SECONDS}s limit")
            profiler = cProfile.Profile() if PROFILE_REQUESTS else None
            if profiler:
                profiler.enable()
            try:
                result = transcribe_audio(model, spec, audio_file, audio_type)
            finally:
                if profiler:
                    profiler.disable()
            if profiler:
                stream = io.StringIO()
                pstats.Stats(profiler, stream=stream).sort_stats("cumulative").print_stats(15)
                print("CPROFILE:\n" + stream.getvalue(), flush=True)
            if not completed.is_set():
                results.put(("ok", result))
        except Exception as exc:
            print(f"REAL ERROR: {type(exc).__name__}: {exc}", flush=True)
            if not completed.is_set():
                results.put(("error", exc))

    thread = threading.Thread(target=worker)
    thread.start()
    try:
        status, result = results.get(timeout=REQUEST_TIMEOUT_SECONDS)
        elapsed_ms = (time.perf_counter() - started) * 1000
        completed.set()
        if status == "ok":
            text, confidence = result
            return text, confidence, elapsed_ms, status, None
        return None, None, elapsed_ms, status, str(result)
    except queue.Empty:
        elapsed_ms = (time.perf_counter() - started) * 1000
        completed.set()
        print("Request timed out. Waiting for its worker before the next attempt (outside recorded latency).", flush=True)
        return None, None, elapsed_ms, "timeout", f"Exceeded {REQUEST_TIMEOUT_SECONDS:g}s request timeout"
    finally:
        # The script leaves timeout workers running. Benchmark attempts must not overlap
        # or free a model while its worker still uses it; the CSV reports timeout latency.
        thread.join()


def normalize_text(text):
    return re.sub(r"[^\w\s]", "", str(text).lower().strip())


RUNTIME_INFO = configure_runtime()
print(f"Torch threads: {RUNTIME_INFO['torch threads']}; priority: {RUNTIME_INFO['process priority']}")
print("Models configured:", [spec["name"] for spec in MODELS])
print("Input:", Path(AUDIO_DIR).resolve())
print("Output:", Path(OUTPUT_CSV).resolve())
print("Attempts per audio/model:", ATTEMPTS_PER_AUDIO)
print("HF Whisper uses checkpoint generation defaults without a keyword prompt; only stock Whisper uses keywords.")


In [ ]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    """Implementation of the 3-step intent matching logic."""
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    # Step 1: Clean text
    clean = normalize_text(transcript)
    words = clean.split()

    # Step 2: Get intents for current state
    state_intents = INTENT_CONFIG.get(state, {})

    # Step 3: Keyword group matching
    for intent_name, groups in state_intents.items():
        for group in groups:
            # Match if EVERY keyword in group is found as a whole word
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown" # Or noaudio if you prefer

In [ ]:
def run_model_comparison(models, output_csv, state, attempts_per_audio=1):
    if type(attempts_per_audio) is not int or attempts_per_audio < 1:
        raise ValueError("ATTEMPTS_PER_AUDIO must be a positive integer.")
    if not models:
        raise ValueError("Add at least one entry to MODELS.")
    names = [spec["name"] for spec in models]
    if any(not name.strip() for name in names) or len(set(names)) != len(names):
        raise ValueError("Every model needs a nonempty, unique name.")
    for spec in models:
        if spec["backend"] not in {"whisper", "huggingface", "nemo"} or not spec["model"]:
            raise ValueError(f"Invalid model configuration: {spec}")
    if state not in {"decision", "reason", "default"}:
        raise ValueError("BATCH_STATE must be decision, reason, or default.")
    if REQUEST_TIMEOUT_SECONDS is not None and REQUEST_TIMEOUT_SECONDS <= 0:
        raise ValueError("REQUEST_TIMEOUT_SECONDS must be positive, or None to wait without a timeout.")

    folder = Path(AUDIO_DIR)
    if not folder.is_dir():
        raise FileNotFoundError(f"Audio folder not found: {folder.resolve()}")
    files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".wav")
    if not files:
        raise ValueError(f"No WAV files found in {folder.resolve()}")
    output_path = Path(output_csv)
    if output_path.resolve() in {p.resolve() for p in Path('.').glob('*_audio_log.csv')}:
        raise ValueError("Use a separate OUTPUT_CSV, not an original audio log.")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Header metadata only: each timed request reads/decodes its file like the script.
    rows = []
    for path in files:
        duration = get_audio_duration_seconds(path)
        for attempt in range(1, attempts_per_audio + 1):
            rows.append({"fileName": path.name, "attempt": attempt, "audio type": state,
                         "audio duration [s]": round(duration, 3) if duration is not None else None})
    df = pd.DataFrame(rows)
    fields = ["model", "transcription", "confidence", "intent [decision]", "intent [reason]",
              "processing time [ms]", "load time [ms]", "warmup time [ms]", "device", "dtype", "status", "error"]
    for spec in models:
        for field in fields:
            df[f"{spec['name']} | {field}"] = None
        model_id = f"{spec['backend']}:{spec['model']}"
        if spec.get("filename"):
            model_id += "/" + spec["filename"]
        df[f"{spec['name']} | model"] = model_id

    runtime = dict(RUNTIME_INFO, **{"audio type": state, "attempts per audio": attempts_per_audio,
                                  "request profiling": PROFILE_REQUESTS,
                                  "request timeout [s]": REQUEST_TIMEOUT_SECONDS, "models": {}})
    metadata_path = output_path.with_suffix(".runtime.json")
    print(f"Comparing {len(models)} models on {len(files)} WAV files, {attempts_per_audio} attempts each.")
    print(f"Reference request timing; torch threads={torch.get_num_threads()}, profiling={PROFILE_REQUESTS}.")
    print("Includes file decoding, confidence and request/profiling overhead; excludes loading, warm-up and intent classification.")
    print("HF Whisper/NeMo are warmed once. Stock Whisper is not warmed, matching pythonChatAgent.py.")

    for spec in models:
        prefix, model, phase = spec["name"] + " | ", None, "load"
        try:
            started = time.perf_counter()
            model = get_model(spec)
            df[prefix + "load time [ms]"] = round((time.perf_counter() - started) * 1000, 2)
            actual_model = model["model"] if spec["backend"] == "huggingface" else model
            parameter = next(actual_model.parameters())
            device, dtype = str(parameter.device), str(parameter.dtype)
            df[prefix + "device"], df[prefix + "dtype"] = device, dtype
            details = {"spec": spec, "device": device, "dtype": dtype}
            if spec["backend"] == "huggingface":
                details["generation config"] = actual_model.generation_config.to_dict()
            runtime["models"][spec["name"]] = details
            del actual_model, parameter
            phase = "warmup"
            df[prefix + "warmup time [ms]"] = round(warmup_model(model, spec), 2)
            print(f"Ready {spec['name']}: {device} / {dtype}; warm-up {df[prefix + 'warmup time [ms]'].iloc[0]:.0f} ms")
            phase = "request"
            for idx, row in df.iterrows():
                text, confidence, elapsed_ms, status, error = run_request(model, spec, folder / row["fileName"], state)
                df.at[idx, prefix + "transcription"] = text
                df.at[idx, prefix + "confidence"] = confidence
                df.at[idx, prefix + "processing time [ms]"] = round(elapsed_ms, 2)
                df.at[idx, prefix + "status"] = status
                df.at[idx, prefix + "error"] = error
                if status == "ok":
                    df.at[idx, prefix + "intent [decision]"] = determine_intent(text, "decision")
                    df.at[idx, prefix + "intent [reason]"] = determine_intent(text, "reason")
                print(f"{spec['name']} / {row['fileName']} / attempt {row['attempt']}: {status} {text!r} ({elapsed_ms:.0f} ms)")
                # Save every attempt so interruption cannot discard earlier requests.
                df.to_csv(output_path, index=False)
        except Exception as exc:
            # Preserve completed attempts if an unexpected request/reporting failure occurs.
            pending = df[prefix + "status"].isna()
            df.loc[pending, prefix + "status"] = phase + "_error"
            df.loc[pending, prefix + "error"] = str(exc)
            print(f"ERROR {phase} {spec['name']}: {exc}")
        finally:
            del model
            gc.collect()
            df.to_csv(output_path, index=False)
            metadata_path.write_text(json.dumps(runtime, indent=2, default=str) + "\n", encoding="utf-8")

    print(f"Saved {len(df)} rows to {output_path.resolve()}")
    print(f"Runtime/package details: {metadata_path.resolve()}")
    display(df)
    return df


comparison_df = run_model_comparison(MODELS, OUTPUT_CSV, BATCH_STATE, ATTEMPTS_PER_AUDIO)


In [ ]:
# Quick timing summary. Raw transcription and both intents are in OUTPUT_CSV.
summary = []
for spec in MODELS:
    prefix = spec["name"] + " | "
    successful = comparison_df[prefix + "status"].eq("ok")
    timings = pd.to_numeric(comparison_df.loc[successful, prefix + "processing time [ms]"])
    summary.append({
        "model": spec["name"],
        "successful attempts": int(successful.sum()),
        "failed attempts": int((~successful).sum()),
        "mean processing time [ms]": round(timings.mean(), 2),
    })
display(pd.DataFrame(summary))
print("Reference request latency; compare the same checkpoint, audio type, hardware and package versions. Timeouts are excluded from means.")
